# Getting Started with CTSM and NEON Tower Sites

This notebook introduces the key components of the containerized CTSM
(Community Terrestrial Systems Model) environment and walks through a
basic setup to confirm everything is working.

**No credentials or data downloads are required.** You can run this
notebook immediately after launching the container.

### What is CTSM?

CTSM is the Community Terrestrial Systems Model, the standalone version
of CLM (Community Land Model). It simulates land surface processes:
soil temperature and moisture, vegetation growth, carbon cycling, snow
dynamics, and energy/water exchange with the atmosphere.

### What is NEON?

The National Ecological Observatory Network operates 81 instrumented
field sites across the US. Each site has a tower that continuously
measures soil conditions, carbon fluxes, and meteorological variables.
CTSM can be configured to run at a specific NEON tower site and compare
its predictions against real observations.

### What this notebook covers

1. **Environment check** -- verify the Python scientific stack is installed
2. **Working with NetCDF** -- the standard file format for climate model output
3. **Map visualization** -- rendering geospatial data with cartopy
4. **CTSM and CIME** -- the model source tree and build system
5. **NEON tower sites** -- discovering available sites
6. **Creating a NEON case** -- setting up a simulation for a specific tower

## 1. Environment Check

The container ships a complete scientific Python stack installed via
conda-forge. This cell verifies that the key packages are available
and prints their versions.

In [ ]:
import sys
print(f"Python {sys.version}")
print(f"Platform: {sys.platform}")
print()

packages = [
    "numpy", "scipy", "pandas", "xarray", "netCDF4",
    "matplotlib", "cartopy", "bokeh", "holoviews", "panel",
    "jupyterlab", "dask", "boto3", "esmpy",
]

for name in packages:
    mod = __import__(name)
    ver = getattr(mod, "__version__", "ok")
    print(f"  {name:15s} {ver}")

print("\nAll imports OK.")

## 2. Working with NetCDF

Climate model output is stored in NetCDF format, a self-describing
binary format widely used in earth sciences. This cell creates a small
synthetic dataset, writes it to a NetCDF file, and reads it back to
verify the I/O libraries (HDF5, NetCDF-C) are linked correctly.

[xarray](https://xarray.dev/) is the standard Python library for working
with labeled multi-dimensional arrays in NetCDF files.

In [ ]:
import numpy as np
import xarray as xr
import tempfile, pathlib

# Create a small synthetic dataset
lats = np.linspace(-90, 90, 19)
lons = np.linspace(0, 360, 36, endpoint=False)
np.random.seed(42)
data = 260 + 30 * np.random.rand(19, 36).astype(np.float32)

ds = xr.Dataset(
    {"temperature": (["lat", "lon"], data)},
    coords={"lat": lats, "lon": lons},
    attrs={"title": "Smoke test dataset"},
)

# Write to NetCDF and read back
with tempfile.TemporaryDirectory() as tmp:
    path = pathlib.Path(tmp) / "test.nc"
    ds.to_netcdf(path)
    ds_read = xr.open_dataset(path)
    assert "temperature" in ds_read
    np.testing.assert_array_almost_equal(
        ds_read["temperature"].values, data, decimal=5
    )
    ds_read.close()

print(f"NetCDF round-trip OK ({path.stat().st_size:,} bytes written).")
ds

## 3. Map Visualization with Cartopy

[Cartopy](https://scitools.org.uk/cartopy/) handles map projections and
geospatial plotting. It is used throughout the project's analysis
notebooks to visualize CLM output fields on maps.

This cell renders two maps: one showing coastlines and national borders,
and one showing the synthetic temperature field from the previous section
projected onto a Robinson globe.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, axes = plt.subplots(
    1, 2, figsize=(14, 4),
    subplot_kw={"projection": ccrs.Robinson()},
)

# Left: coastlines and borders
ax = axes[0]
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#e8e8e8")
ax.add_feature(cfeature.OCEAN, facecolor="#d0e4f0")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
ax.set_title("Coastlines + Borders")

# Right: pcolormesh of the synthetic temperature data
ax = axes[1]
ax.set_global()
lon2d, lat2d = np.meshgrid(lons, lats)
im = ax.pcolormesh(
    lon2d, lat2d, data,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r", vmin=260, vmax=290,
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.set_title("Synthetic Temperature Field")
plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, label="K")

plt.tight_layout()
plt.show()
print("Cartopy rendering OK.")

## 4. CTSM and CIME

The container includes the full CTSM source tree at `/opt/ncar/ctsm`.
Inside it, **CIME** (Common Infrastructure for Modeling the Earth) is
the build and case management system. You interact with CIME through
scripts like:

- `create_newcase` -- create a new simulation case
- `case.setup` -- generate build scripts and namelists
- `case.build` -- compile the Fortran model
- `case.submit` -- run the simulation

This cell verifies that CTSM and CIME are accessible.

In [ ]:
import os
import subprocess

ctsm_root = os.environ.get("CESMROOT", "/opt/ncar/ctsm")
print(f"CTSM root:    {ctsm_root}")
print(f"CIME machine: {os.environ.get('CIME_MACHINE', 'not set')}")
print()

# Verify key scripts exist
for script in ["create_newcase", "query_config"]:
    path = os.path.join(ctsm_root, "cime", "scripts", script)
    exists = os.path.exists(path)
    print(f"  {script:20s} {'OK' if exists else 'MISSING'}")

# Verify CTSM Python modules
from ctsm import add_cime_to_path
from ctsm.path_utils import path_to_ctsm_root
print(f"\n  ctsm.path_to_ctsm_root() = {path_to_ctsm_root()}")
print("\nCTSM/CIME availability OK.")

## 5. NEON Tower Sites

Each NEON site has a **usermods** directory in the CTSM source tree that
contains site-specific configuration: which grid cell the tower sits in,
local soil and vegetation parameters, surface dataset paths, and namelist
overrides. These usermods are what make it possible to run CLM at a
specific real-world location with a single command.

This cell scans the usermods directory and lists all available sites.

In [ ]:
import glob

neon_dir = os.path.join(ctsm_root, "cime_config", "usermods_dirs", "clm", "NEON")
sites = sorted([
    os.path.basename(d)
    for d in glob.glob(os.path.join(neon_dir, "[!d]*"))
    if os.path.isdir(d)
])

print(f"NEON usermods directory: {neon_dir}")
print(f"Sites found: {len(sites)}")
print()

# Display in columns
cols = 8
for i in range(0, len(sites), cols):
    print("  ".join(f"{s:6s}" for s in sites[i:i+cols]))

assert len(sites) >= 40, f"Expected 40+ NEON sites, found {len(sites)}"
print(f"\nNEON site discovery OK ({len(sites)} sites).")

## 6. Creating a NEON Case

This cell walks through creating a CTSM case for the **KONZ** (Konza
Prairie, Kansas) NEON site. Konza Prairie is a tallgrass prairie
ecological research site, one of the most-studied NEON locations.

The case is created with `--setup-only` mode, which configures the
simulation without downloading input data or compiling Fortran code.
This demonstrates the CIME workflow end-to-end in a few seconds.

To actually run a simulation, you would continue with `case.build`
(compile the model, ~2 min) and `case.submit` (run, requires input data).

In [ ]:
%%bash
set -e

SITE="KONZ"
OUTPUT_ROOT="/tmp/smoke_test_neon"
rm -rf "$OUTPUT_ROOT"
mkdir -p /home/user/inputdata /home/user/scratch

echo "Creating NEON case for site: $SITE"
echo "Output root: $OUTPUT_ROOT"
echo

# Create case (setup only, no build or run)
$CESMROOT/cime/scripts/create_newcase \
    --case "$OUTPUT_ROOT/$SITE" \
    --compset I1PtClm60Bgc \
    --res CLM_USRDAT \
    --machine container \
    --run-unsupported \
    --user-mods-dirs "$CESMROOT/cime_config/usermods_dirs/clm/NEON/$SITE" \
    2>&1 | tail -5

echo
echo "Running case.setup..."
cd "$OUTPUT_ROOT/$SITE" && ./case.setup 2>&1 | tail -3

echo
echo "Case directory contents:"
ls "$OUTPUT_ROOT/$SITE/" | head -10

echo
echo "NEON case creation OK."

# Clean up
rm -rf "$OUTPUT_ROOT"

## Next Steps

Everything is working. From here you can:

- **Run a NEON tower simulation** -- see the `Design_Hub_v2` and
  `Modeling_Hub` notebooks for full workflows that run CLM at NEON
  sites and compare output against observations.

- **Analyze existing model output** -- the `Data_Hub` notebook
  demonstrates loading CLM history files from S3, computing diagnostics,
  and plotting soil profiles. (Requires S3 credentials.)

- **Build the Fortran model** -- to compile CTSM for a specific case,
  run `case.build` after `case.setup`. The automated test suite
  validates this:
  ```bash
  ./tests/run_container_tests.sh tier2
  ```

- **Learn more about CTSM** -- the
  [CTSM documentation](https://escomp.github.io/CTSM/) and
  [NCAR CTSM Tutorial](https://github.com/NCAR/CTSM-Tutorial) are
  excellent resources.